# 07 — Recommendations
**Purpose:** Synthesise all analysis into prioritised, actionable recommendations.
**Inputs:** All intermediate outputs from notebooks 02–06.
**Outputs:** `data/intermediate/recommendations.parquet`, `data/intermediate/recommendations_summary.txt`

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from buildanalysis.build import compute_critical_path
from buildanalysis.graphs import build_dependency_graph
from buildanalysis.loading import BuildDataset
from buildanalysis.recommend import (
    Intervention,
    InterventionType,
    build_pareto_frontier,
    format_recommendation_summary,
    score_dependency_interventions,
    score_header_interventions,
    score_target_split_interventions,
)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 100})

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

## 1. Load all intermediate results

In [ ]:
# Primary data
tm = ds.target_metrics
el = ds.edge_list
fm = ds.file_metrics

# Build graph and critical path (needed by several intervention scorers)
bg = build_dependency_graph(tm, el, direct_only=True)
cp_result = compute_critical_path(bg, tm)

print(f"Targets: {len(tm):,}  |  Edges: {len(el):,}  |  Files: {len(fm):,}")
print(f"Critical path: {cp_result.total_time_s:.1f}s  ({len(cp_result.path)} targets)")
print(f"Parallelism ratio: {cp_result.parallelism_ratio:.1f}x")

In [ ]:
# Intermediate results from prior notebooks — load gracefully
intermediates: dict[str, pd.DataFrame | None] = {}

for name in [
    "target_features",       # notebook 02
    "critical_path",         # notebook 03
    "header_impact",         # notebook 04
    "graph_analysis",        # notebook 05
    "communities",           # notebook 06
    "feature_configurations",# notebook 06
]:
    try:
        intermediates[name] = ds.load_intermediate(name)
        print(f"  Loaded {name}: {intermediates[name].shape}")
    except FileNotFoundError:
        intermediates[name] = None
        print(f"  MISSING {name} — skipping related interventions")

## 2. Header interventions

In [ ]:
header_interventions: list[Intervention] = []

header_impact = intermediates["header_impact"]
if header_impact is not None:
    # score_header_interventions expects include_amplification — pass empty if unavailable
    include_amp = pd.DataFrame(columns=["file", "direct_includes", "transitive_includes", "amplification_ratio"])

    header_interventions = score_header_interventions(
        header_impact=header_impact,
        include_amplification=include_amp,
        top_n=20,
    )

    total_impact_s = sum(iv.estimated_build_time_reduction_ms for iv in header_interventions) / 1000
    print(f"Header interventions: {len(header_interventions)} candidates")
    print(f"Total estimated impact: {total_impact_s:.1f}s")

    for iv in header_interventions[:5]:
        print(f"  - {iv.description}  ({iv.estimated_build_time_reduction_ms:.0f}ms, ~{iv.estimated_effort_days:.0f}d)")
else:
    print("No header_impact data — skipping header interventions.")

## 3. Dependency interventions

In [ ]:
dep_interventions: list[Intervention] = []

# score_dependency_interventions expects 'source' and 'dependency' columns
el_renamed = el.rename(columns={"source_target": "source", "dest_target": "dependency"})

dep_interventions = score_dependency_interventions(
    bg=bg,
    timing=tm,
    critical_path_result=cp_result,
    edge_list=el_renamed,
    top_n=20,
)

total_impact_s = sum(iv.estimated_build_time_reduction_ms for iv in dep_interventions) / 1000
print(f"Dependency interventions: {len(dep_interventions)} candidates")
print(f"Total estimated impact: {total_impact_s:.1f}s")

has_visibility = "cmake_visibility" in el.columns
if has_visibility:
    n_narrow = sum(1 for iv in dep_interventions if iv.intervention_type == InterventionType.NARROW_VISIBILITY)
    n_remove = len(dep_interventions) - n_narrow
    print(f"  Visibility narrowing: {n_narrow}  |  Edge removal: {n_remove}")

for iv in dep_interventions[:5]:
    print(f"  - {iv.description}  ({iv.estimated_build_time_reduction_ms:.0f}ms)")

## 4. Target-splitting interventions

In [ ]:
split_interventions = score_target_split_interventions(
    target_metrics=tm,
    critical_path_result=cp_result,
    top_n=10,
)

total_impact_s = sum(iv.estimated_build_time_reduction_ms for iv in split_interventions) / 1000
print(f"Target-splitting interventions: {len(split_interventions)} candidates")
print(f"Total estimated impact: {total_impact_s:.1f}s")

for iv in split_interventions[:5]:
    print(f"  - {iv.description}  ({iv.estimated_build_time_reduction_ms:.0f}ms, ~{iv.estimated_effort_days:.0f}d)")

## 5. Codegen interventions

In [ ]:
codegen_interventions: list[Intervention] = []

# Identify targets with high codegen ratio and significant codegen time
codegen_candidates = tm[
    (tm["codegen_ratio"] > 0.3) & (tm["codegen_time_ms"].notna()) & (tm["codegen_time_ms"] > 0)
].sort_values("codegen_time_ms", ascending=False)

cp_set = set(cp_result.path)

for _, row in codegen_candidates.head(10).iterrows():
    target = row["cmake_target"]
    codegen_ms = row["codegen_time_ms"]
    on_cp = target in cp_set

    # Cross-reference with git churn: low-churn codegen inputs are good caching candidates
    churn = row.get("git_churn_total", 0)
    if churn < 50:
        confidence = 0.7
        rationale = (
            f"Target has {row['codegen_ratio']:.0%} generated code and {codegen_ms:.0f}ms codegen time. "
            f"Low git churn ({churn}) suggests inputs are stable — caching is viable."
        )
    else:
        confidence = 0.5
        rationale = (
            f"Target has {row['codegen_ratio']:.0%} generated code and {codegen_ms:.0f}ms codegen time. "
            f"Higher git churn ({churn}) — caching benefit depends on input stability."
        )

    codegen_interventions.append(
        Intervention(
            intervention_type=InterventionType.CACHE_CODEGEN,
            description=f"Cache codegen for {target}",
            targets_affected=[target],
            estimated_build_time_reduction_ms=codegen_ms * 0.8,
            estimated_effort_days=max(1.0, min(5.0, row["codegen_file_count"] * 0.3)),
            confidence=confidence,
            rationale=rationale,
        )
    )

print(f"Codegen interventions: {len(codegen_interventions)} candidates")
if codegen_interventions:
    total_impact_s = sum(iv.estimated_build_time_reduction_ms for iv in codegen_interventions) / 1000
    print(f"Total estimated impact: {total_impact_s:.1f}s")
    for iv in codegen_interventions[:5]:
        print(f"  - {iv.description}  ({iv.estimated_build_time_reduction_ms:.0f}ms)")
else:
    print("No targets with significant codegen overhead found.")

## 6. Pareto frontier

In [ ]:
# Collect all interventions
all_interventions = header_interventions + dep_interventions + split_interventions + codegen_interventions
print(f"Total interventions collected: {len(all_interventions)}")

pareto_df = build_pareto_frontier(all_interventions)
n_pareto = pareto_df["pareto_optimal"].sum()
print(f"Pareto-optimal interventions: {n_pareto}")

In [ ]:
# Scatter: impact vs effort, highlight Pareto-optimal
if not pareto_df.empty:
    fig, ax = plt.subplots(figsize=(12, 7))

    non_pareto = pareto_df[~pareto_df["pareto_optimal"]]
    optimal = pareto_df[pareto_df["pareto_optimal"]]

    # Map intervention types to colours
    type_colours = {
        "REFACTOR_HEADER": "steelblue",
        "REMOVE_DEPENDENCY": "coral",
        "NARROW_VISIBILITY": "orange",
        "SPLIT_TARGET": "teal",
        "CACHE_CODEGEN": "mediumpurple",
        "ADD_PCH": "olive",
        "FORWARD_DECLARE": "pink",
        "EXTRACT_INTERFACE": "brown",
    }

    for _, row in non_pareto.iterrows():
        ax.scatter(
            row["effort_days"], row["impact_ms"] / 1000,
            c=type_colours.get(row["type"], "grey"), alpha=0.4, s=50, zorder=2,
        )

    for _, row in optimal.iterrows():
        ax.scatter(
            row["effort_days"], row["impact_ms"] / 1000,
            c=type_colours.get(row["type"], "grey"), edgecolors="black", linewidths=1.5,
            s=120, zorder=3,
        )

    # Draw Pareto frontier line
    frontier = optimal.sort_values("effort_days")
    ax.step(
        frontier["effort_days"], frontier["impact_ms"] / 1000,
        where="post", color="black", linestyle="--", alpha=0.5, zorder=1,
    )

    # Legend for types
    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=t.replace("_", " ").title())
        for t, c in type_colours.items()
        if t in pareto_df["type"].values
    ]
    handles.append(Line2D([0], [0], marker="o", color="w", markerfacecolor="grey",
                          markeredgecolor="black", markeredgewidth=1.5, markersize=10, label="Pareto optimal"))
    ax.legend(handles=handles, loc="upper right", fontsize=9)

    ax.set_xlabel("Estimated effort (days)")
    ax.set_ylabel("Estimated build time reduction (s)")
    ax.set_title("Intervention Impact vs Effort — Pareto Frontier")
    fig.tight_layout()
    plt.show()
else:
    print("No interventions to plot.")

In [ ]:
# Pareto-optimal interventions table
if not pareto_df.empty:
    optimal = pareto_df[pareto_df["pareto_optimal"]].copy()
    display_cols = ["type", "description", "impact_ms", "effort_days", "confidence", "rationale"]
    optimal["impact_s"] = optimal["impact_ms"] / 1000
    print(f"PARETO-OPTIMAL INTERVENTIONS ({len(optimal)})")
    print("=" * 120)
    print(optimal[["type", "description", "impact_s", "effort_days", "confidence"]].to_string(index=False))

## 7. Global recommendation summary

In [ ]:
summary_text = format_recommendation_summary(pareto_df, top_n=10)
print(summary_text)

In [ ]:
# Stakeholder prose summary
cp_time_min = cp_result.total_time_s / 60
top3 = pareto_df[pareto_df["pareto_optimal"]].head(3)
top3_reduction_s = top3["impact_ms"].sum() / 1000
top3_reduction_min = top3_reduction_s / 60
top3_pct = (top3_reduction_s / cp_result.total_time_s * 100) if cp_result.total_time_s > 0 else 0

top1_desc = pareto_df.iloc[0]["description"] if len(pareto_df) > 0 else "N/A"

prose = (
    f"The build critical path is {cp_time_min:.1f} minutes. "
    f"The top 3 interventions could reduce this by {top3_reduction_min:.1f} minutes "
    f"({top3_pct:.0f}%). The most impactful single change is: {top1_desc}."
)
print("\n--- Stakeholder Summary ---")
print(prose)

## 8. Modularity recommendations

In [ ]:
feature_configs = intermediates["feature_configurations"]
communities = intermediates["communities"]

if feature_configs is not None:
    print("MODULARITY ANALYSIS")
    print("=" * 80)

    # Which features are viable for independent builds?
    viable = feature_configs[feature_configs["build_fraction"] < 0.4]
    print(f"\nViable independent features (build_fraction < 40%): {len(viable)} / {len(feature_configs)}")
    if not viable.empty:
        for _, row in viable.iterrows():
            print(f"  Feature {row['feature_id']}: "
                  f"{row['own_targets']} own targets, "
                  f"{row['total_build_set']} total build set, "
                  f"build fraction = {row['build_fraction']:.1%}")

    # Shared core size
    if "total_build_set" in feature_configs.columns and "own_targets" in feature_configs.columns:
        total_targets = len(tm)
        mean_transitive_deps = feature_configs["transitive_deps"].mean()
        core_pct = mean_transitive_deps / total_targets * 100 if total_targets > 0 else 0
        print(f"\nShared core (avg transitive deps per feature): "
              f"{mean_transitive_deps:.0f} targets ({core_pct:.0f}% of codebase)")

    # Cross-community dependency analysis
    if communities is not None and "community" in communities.columns:
        community_map = communities.set_index("cmake_target")["community"].to_dict()
        cross_edges = el[
            el.apply(lambda r: community_map.get(r["source_target"]) != community_map.get(r["dest_target"]), axis=1)
        ]
        total_edges = len(el)
        cross_pct = len(cross_edges) / total_edges * 100 if total_edges > 0 else 0
        print(f"\nCross-community edges: {len(cross_edges)} / {total_edges} ({cross_pct:.0f}%)")

        # Top cross-community dependency pairs
        if not cross_edges.empty:
            cross_edges = cross_edges.copy()
            cross_edges["src_community"] = cross_edges["source_target"].map(community_map)
            cross_edges["dst_community"] = cross_edges["dest_target"].map(community_map)
            pair_counts = (
                cross_edges.groupby(["src_community", "dst_community"])
                .size()
                .reset_index(name="n_edges")
                .sort_values("n_edges", ascending=False)
            )
            print("\nTop cross-community dependency pairs:")
            for _, row in pair_counts.head(10).iterrows():
                print(f"  Community {row['src_community']} -> {row['dst_community']}: {row['n_edges']} edges")

    print("\n--- Recommended phased approach ---")
    print("  Phase 1: Establish the shared core as a stable, well-defined base library set.")
    print("  Phase 2: Split the most independent features into separate build targets/repos.")
    print("  Phase 3: Address cross-feature coupling to enable further feature separation.")
else:
    print("No feature_configurations data — skipping modularity recommendations.")

## 9. Quick wins vs strategic investments

In [ ]:
if not pareto_df.empty:
    df = pareto_df.copy()
    df["impact_s"] = df["impact_ms"] / 1000

    def categorise(row):
        if row["effort_days"] < 2 and row["confidence"] > 0.6:
            return "Quick win (<2 days)"
        elif row["effort_days"] <= 10:
            return "Medium investment (2-10 days)"
        else:
            return "Strategic investment (>10 days)"

    df["category"] = df.apply(categorise, axis=1)

    for cat in ["Quick win (<2 days)", "Medium investment (2-10 days)", "Strategic investment (>10 days)"]:
        subset = df[df["category"] == cat].sort_values("impact_ms", ascending=False)
        print(f"\n{cat.upper()}: {len(subset)} interventions")
        print("-" * 100)
        if subset.empty:
            print("  (none)")
        else:
            print(subset[["type", "description", "impact_s", "effort_days", "confidence"]].to_string(index=False))
else:
    print("No interventions to categorise.")

## 10. Persist output

In [ ]:
# Save recommendations parquet
out_path = ds.save_intermediate("recommendations", pareto_df)
print(f"Saved recommendations.parquet: {pareto_df.shape[0]} rows x {pareto_df.shape[1]} cols")
print(f"  -> {out_path}")

# Save formatted summary text
summary_path = INTERMEDIATE_DIR / "recommendations_summary.txt"
summary_path.write_text(summary_text + "\n\n" + prose + "\n")
print("\nSaved recommendations_summary.txt")
print(f"  -> {summary_path}")